In [1]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="sk-no-key-required",
)

resp = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "Отвечай кратко и по делу."},
        {"role": "user", "content": "Что такое градиентный бустинг?"}
    ],
    temperature=0.2,
    max_tokens=200,
)

print(resp.choices[0].message.content)

Градиентный бустинг - это метод машинного обучения, основанный на комбинации нескольких слабых моделей (обычно случайных) для повышения точности.


In [4]:
import os
from typing import List

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer

from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_huggingface import HuggingFacePipeline
from langchain_community.vectorstores import FAISS



/home/student/llm-u/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# ============================================================
# 1. Пути к твоим локальным моделям
# ============================================================
LLM_PATH = "./models/llm"
EMB_PATH = "./models/emb"

In [7]:
# ============================================================
# 2. Своя embedding-модель для LangChain
# ============================================================
class LocalSentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_path: str, device: str = None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_path, device=device)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        embeddings = self.model.encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        return embeddings.tolist()

    def embed_query(self, text: str) -> List[float]:
        embedding = self.model.encode(
            text,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        return embedding.tolist()


# ============================================================
# 3. Загружаем свою LLM с диска
# ============================================================
def build_local_llm(model_path: str):
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )

    text_gen_pipeline = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.0,
        repetition_penalty=1.05,
        return_full_text=False,
    )

    llm = HuggingFacePipeline(pipeline=text_gen_pipeline)
    return llm


# ============================================================
# 4. Небольшая база документов
# ============================================================
docs = [
    Document(
        page_content=(
            "LangChain позволяет строить пайплайны из prompt, моделей, "
            "retriever и output parser."
        ),
        metadata={"source": "doc_1"},
    ),
    Document(
        page_content=(
            "FAISS — это библиотека для поиска ближайших соседей по векторным "
            "представлениям. Часто используется в RAG."
        ),
        metadata={"source": "doc_2"},
    ),
    Document(
        page_content=(
            "Embedding-модель переводит текст в вектор. Затем по этим векторам "
            "можно искать похожие документы."
        ),
        metadata={"source": "doc_3"},
    ),
    Document(
        page_content=(
            "Если использовать локальные модели, то LangChain может работать "
            "полностью без OpenAI API."
        ),
        metadata={"source": "doc_4"},
    ),
]



In [8]:

# ============================================================
# 5. Создаём embeddings и индекс
# ============================================================
embeddings = LocalSentenceTransformerEmbeddings(model_path=EMB_PATH)
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


No sentence-transformers model found with name ./models/emb. Creating a new one with mean pooling.


ValueError: Unrecognized model in ./models/emb. Should have a `model_type` key in its config.json.

In [ ]:
# ============================================================
# 6. Загружаем локальную LLM
# ============================================================
llm = build_local_llm(LLM_PATH)



In [ ]:

# ============================================================
# 7. Prompt для RAG
# ============================================================
prompt = ChatPromptTemplate.from_template(
    """Ты помощник, который отвечает только на основе контекста.

Контекст:
{context}

Вопрос:
{question}

Дай краткий и понятный ответ по-русски."""
)

output_parser = StrOutputParser()



In [ ]:

# ============================================================
# 8. Функция сборки контекста
# ============================================================
def format_docs(retrieved_docs: List[Document]) -> str:
    parts = []
    for i, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get("source", f"doc_{i}")
        parts.append(f"[{source}]\n{doc.page_content}")
    return "\n\n".join(parts)



In [ ]:
# ============================================================
# 9. Один запрос: retrieval -> prompt -> LLM
# ============================================================
question = "Объясни простыми словами, как LangChain работает с локальными моделями"

retrieved_docs = retriever.invoke(question)
context = format_docs(retrieved_docs)

chain = prompt | llm | output_parser
answer = chain.invoke({
    "context": context,
    "question": question,
})

print("=== Найденные документы ===")
for doc in retrieved_docs:
    print(f"- {doc.metadata['source']}: {doc.page_content}")

print("\n=== Ответ модели ===")
print(answer)